### 1) Configuration

In [0]:
%pip install faker

In [0]:
# ==========================================================
# Enterprise Multi-Source Generator
# Configuration
# ==========================================================

from faker import Faker
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime, timedelta
import pandas as pd
import random
import uuid
import numpy as np

# ------------------------------------------------------------------
# Seeds (reproducible generation)
# ------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)

fake = Faker("pt_BR")

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

LANDING_ROOT = "/Volumes/landing/landing/landing"
OLIST_ROOT = "/Volumes/landing/landing/olist_dev"

LOAD_DATE = "2026-07-21"

YEAR, MONTH, DAY = LOAD_DATE.split("-")
DATE_PATH = f"{YEAR}/{MONTH}/{DAY}"

# ------------------------------------------------------------------
# Source Systems Distribution
# ------------------------------------------------------------------

SYSTEM_DISTRIBUTION = {
    "crm": 1.00,          # 100%
    "erp": 0.90,          # 90%
    "ecommerce": 0.80,    # 80%
    "loyalty": 0.35       # 35%
}

# ------------------------------------------------------------------
# Output folders
# ------------------------------------------------------------------

OUTPUT_PATHS = {
    "crm_customers":
        f"{LANDING_ROOT}/crm/customers/{DATE_PATH}",

    "erp_customers":
        f"{LANDING_ROOT}/erp/customers/{DATE_PATH}",

    "erp_orders":
        f"{LANDING_ROOT}/erp/orders/{DATE_PATH}",

    "erp_payments":
        f"{LANDING_ROOT}/erp/payments/{DATE_PATH}",

    "ecommerce_customers":
        f"{LANDING_ROOT}/ecommerce/customers/{DATE_PATH}",

    "ecommerce_products":
        f"{LANDING_ROOT}/ecommerce/products/{DATE_PATH}",

    "ecommerce_sellers":
        f"{LANDING_ROOT}/ecommerce/sellers/{DATE_PATH}",

    "loyalty_customers":
        f"{LANDING_ROOT}/loyalty/customers/{DATE_PATH}"
}

# ------------------------------------------------------------------
# Data Quality Percentages
# ------------------------------------------------------------------

DQ = {

    "uppercase_name": 0.15,

    "abbreviated_name": 0.20,

    "missing_phone": 0.05,

    "missing_email": 0.03,

    "email_typo": 0.02,

    "missing_middle_name": 0.30

}

print("Configuration loaded.")

### 2) Read Olist & Initial Profiling

In [0]:
# ==========================================================
# Read Olist Datasets
# ==========================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Reading Olist datasets...")
print("=" * 80)

customers = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/customers")
)

orders = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/orders")
)

order_items = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/order_items")
)

payments = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/order_payments")
)

reviews = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/order_reviews")
)

products = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/products")
)

sellers = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/sellers")
)

geolocation = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/geolocation")
)

categories = (
    spark.read
        .option("header", True)
        .csv(f"{OLIST_ROOT}/product_category_translation")
)

print("Datasets loaded successfully.")

### Cell 3 â€” Dataset Profiling

In [0]:
# ==========================================================
# Dataset Profiling
# ==========================================================

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "categories": categories
}

summary = []

for name, df in datasets.items():

    summary.append(

        (
            name,
            df.count(),
            len(df.columns)
        )

    )

summary_df = spark.createDataFrame(
    summary,
    ["dataset", "rows", "columns"]
)

display(summary_df)

### Cell 4 â€” Validate Business Keys

In [0]:
# ==========================================================
# Business Key Validation
# ==========================================================

print("=" * 80)
print("Customer statistics")
print("=" * 80)

display(

    customers.select(

        F.count("*").alias("rows"),

        F.countDistinct("customer_id").alias("customer_id"),

        F.countDistinct("customer_unique_id").alias("customer_unique_id")

    )

)

print("=" * 80)
print("Order statistics")
print("=" * 80)

display(

    orders.select(

        F.count("*").alias("rows"),

        F.countDistinct("order_id").alias("order_id"),

        F.countDistinct("customer_id").alias("customer_id")

    )

)

In [0]:
# ==========================================================
# Build Master Customer Profile
# ==========================================================

print("=" * 80)
print("Creating Master Customer Profile...")
print("=" * 80)

# Only one record per real customer

master_seed = (
    customers
    .select(
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates(["customer_unique_id"])
)

master_pd = master_seed.toPandas()
master_pd = master_pd.sort_values("customer_unique_id").reset_index(drop=True)

print(f"Pandas rows: {len(master_pd):,}")

In [0]:
# ==========================================================
# Generate Canonical Customer Information
# ==========================================================

from faker import Faker
import random
import pandas as pd
import uuid

fake = Faker("pt_BR")

genders = ["Male", "Female"]

master_records = []

for row in master_pd.itertuples(index=False):

    first = fake.first_name()

    middle = (
        fake.first_name()
        if random.random() < 0.40
        else None
    )

    last = fake.last_name()

    full = " ".join(
        [x for x in [first, middle, last] if x]
    )

    master_records.append({

        "global_customer_id":
            str(uuid.uuid4()),

        "customer_unique_id":
            row.customer_unique_id,

        "first_name":
            first,

        "middle_name":
            middle,

        "last_name":
            last,

        "full_name":
            full,

        "birth_date":
            fake.date_of_birth(
                minimum_age=18,
                maximum_age=80
            ),

        "gender":
            random.choice(genders),

        "cpf":
            fake.cpf(),

        "personal_email":
            fake.email(),

        "phone":
            fake.cellphone_number(),

        "street":
            fake.street_address(),

        "neighborhood":
            fake.bairro(),

        "city":
            row.customer_city,

        "state":
            row.customer_state

    })

master_profile_pd = pd.DataFrame(master_records)

# Generate sequential source-system IDs on the driver. The previous global
# Spark window moved every customer to one partition and removed parallelism.
seq = pd.Series(range(1, len(master_profile_pd) + 1), index=master_profile_pd.index)
master_profile_pd["crm_customer_id"] = "CRM" + seq.astype(str).str.zfill(7)
master_profile_pd["erp_customer_id"] = "ERP" + seq.astype(str).str.zfill(7)
master_profile_pd["ecommerce_customer_id"] = "WEB" + seq.astype(str).str.zfill(7)
master_profile_pd["loyalty_customer_id"] = "LOY" + seq.astype(str).str.zfill(7)

print(master_profile_pd.head())

In [0]:
# ==========================================================
# Convert Master Profile Back to Spark
# ==========================================================

master_profile = spark.createDataFrame(master_profile_pd).cache()

# Preview removed: the count below materializes the cached profile once.

print("=" * 80)

print(f"Master Customers : {master_profile.count():,}")

print("=" * 80)

In [0]:
# ==========================================================
# Generate System IDs
# ==========================================================

# IDs are generated in Pandas before the single conversion to Spark.
# This intentionally avoids Window.orderBy() without partitionBy().

In [0]:
# ==========================================================
# Helper Functions
# Name Variations
# ==========================================================

import random
import string

def build_full_name(first, middle, last):

    parts = [first]

    if middle:
        parts.append(middle)

    parts.append(last)

    return " ".join(parts)


def random_name(first, middle, last):

    names = []

    names.append(build_full_name(first, middle, last))

    names.append(f"{first} {last}")

    if middle:
        names.append(f"{first} {middle[0]}. {last}")

    names.append(f"{first} {last[0]}.")

    names.append(f"{first[0]}. {last}")

    names.append(f"{first[0]} {last}")

    names.append(f"{last}, {first}")

    names.append(first)

    names.append(last)

    names.append(
        f"{first.lower()}{last.lower()}"
    )

    names.append(
        f"{first.lower()}_{last.lower()}"
    )

    names.append(
        f"{first.lower()}{random.randint(10,999)}"
    )

    return random.choice(names)

In [0]:
# ==========================================================
# Email Generator
# ==========================================================

FREE_EMAILS = [

    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com.br"

]


def random_email(first, last):

    domain = random.choice(FREE_EMAILS)

    options = [

        f"{first}.{last}@{domain}",

        f"{first}{last}@{domain}",

        f"{first[0]}{last}@{domain}",

        f"{first}{random.randint(10,999)}@{domain}",

        f"{first.lower()}_{last.lower()}@{domain}"

    ]

    return random.choice(options).lower()

In [0]:
# ==========================================================
# Phone Formats
# ==========================================================

def phone_variation(phone):

    phone = "".join(filter(str.isdigit, phone))

    if len(phone) < 11:
        return phone

    formats = [

        f"+55 ({phone[:2]}) {phone[2:7]}-{phone[7:]}",

        f"({phone[:2]}) {phone[2:7]}-{phone[7:]}",

        f"{phone}",

        f"{phone[:2]} {phone[2:]}"

    ]

    return random.choice(formats)

In [0]:
# ==========================================================
# Address Formatting
# ==========================================================

def address_variation(address):

    options = [

        address,

        address.replace("Rua","R."),

        address.replace("Avenida","Av."),

        address.replace("Travessa","Tv."),

        address.upper(),

        address.lower()

    ]

    return random.choice(options)

In [0]:
# ==========================================================
# Data Quality Simulation
# ==========================================================

def inject_name_issue(name):

    if random.random() < DQ["uppercase_name"]:

        return name.upper()

    if random.random() < DQ["abbreviated_name"]:

        pieces = name.split()

        if len(pieces) >= 2:

            return f"{pieces[0]} {pieces[-1][0]}."

    return name


def inject_email_issue(email):

    if random.random() > DQ["email_typo"]:

        return email

    return email.replace("@","@@",1)


def maybe_null(value, probability):

    if random.random() < probability:

        return None

    return value

In [0]:
# ==========================================================
# Customer Distribution
# ==========================================================

crm_seed = master_profile

erp_seed = master_profile.sample(
    False,
    SYSTEM_DISTRIBUTION["erp"],
    seed=SEED
)

ecommerce_seed = master_profile.sample(
    False,
    SYSTEM_DISTRIBUTION["ecommerce"],
    seed=SEED
)

loyalty_seed = master_profile.sample(
    False,
    SYSTEM_DISTRIBUTION["loyalty"],
    seed=SEED
)

print("CRM       :", crm_seed.count())
print("ERP       :", erp_seed.count())
print("WEB       :", ecommerce_seed.count())
print("LOYALTY   :", loyalty_seed.count())

In [0]:
# ==========================================================
# Generate CRM Dataset
# ==========================================================

print("=" * 80)
print("Generating CRM...")
print("=" * 80)

crm_pd = crm_seed.toPandas()

crm_records = []

customer_status = [
    "ACTIVE",
    "INACTIVE",
    "PROSPECT"
]

marital_status = [
    "Single",
    "Married",
    "Divorced",
    "Widowed"
]

for c in crm_pd.itertuples(index=False):

    crm_records.append({

        "crm_customer_id": c.crm_customer_id,

        "customer_unique_id": c.customer_unique_id,

        "first_name": c.first_name,

        "middle_name": c.middle_name,

        "last_name": c.last_name,

        "full_name": build_full_name(
            c.first_name,
            c.middle_name,
            c.last_name
        ),

        "cpf": c.cpf,

        "birth_date": c.birth_date,

        "gender": c.gender,

        "marital_status": random.choice(marital_status),

        "profession": fake.job(),

        "annual_income": random.randint(
            25000,
            350000
        ),

        "email": random_email(
            c.first_name,
            c.last_name
        ),

        "phone": phone_variation(
            c.phone
        ),

        "street": address_variation(
            c.street
        ),

        "city": c.city,

        "state": c.state,

        "status": random.choice(customer_status),

        "risk_score": random.randint(0,100),

        "created_at": fake.date_between(
            start_date="-10y",
            end_date="-1y"
        ),

        "updated_at": fake.date_between(
            start_date="-365d",
            end_date="today"
        )

    })

crm_pd = pd.DataFrame(crm_records)

print(crm_pd.head())

In [0]:
# ==========================================================
# CRM Data Quality
# ==========================================================

for i in crm_pd.index:

    crm_pd.at[i,"full_name"] = inject_name_issue(
        crm_pd.at[i,"full_name"]
    )

    crm_pd.at[i,"email"] = inject_email_issue(
        crm_pd.at[i,"email"]
    )

    crm_pd.at[i,"phone"] = maybe_null(
        crm_pd.at[i,"phone"],
        DQ["missing_phone"]
    )

    crm_pd.at[i,"middle_name"] = maybe_null(
        crm_pd.at[i,"middle_name"],
        DQ["missing_middle_name"]
    )

In [0]:
# ==========================================================
# CRM Spark DataFrame
# ==========================================================

crm = spark.createDataFrame(crm_pd)

print("=" * 80)

print("CRM Rows")

print(crm.count())

print("=" * 80)

display(crm.limit(10))

In [0]:
# ==========================================================
# ERP Metrics
# ==========================================================

print("=" * 80)
print("Building ERP Metrics...")
print("=" * 80)

payments_value = (

    payments

    .groupBy("order_id")

    .agg(

        F.sum(
            F.col("payment_value").cast("double")
        ).alias("order_value")

    )

)

erp_metrics = (

    orders

    .join(
        payments_value,
        "order_id",
        "left"
    )

    .join(
        customers.select(
            "customer_id",
            "customer_unique_id"
        ),
        "customer_id"
    )

    .groupBy("customer_unique_id")

    .agg(

        F.countDistinct("order_id")
            .alias("total_orders"),

        F.min("order_purchase_timestamp")
            .alias("first_purchase"),

        F.max("order_purchase_timestamp")
            .alias("last_purchase"),

        F.sum("order_value")
            .alias("total_spent"),

        F.avg("order_value")
            .alias("average_ticket")

    )

).cache()

# Cached because the same aggregation feeds both ERP and Loyalty.

In [0]:
# ==========================================================
# ERP Base
# ==========================================================

erp_base = (

    erp_seed

    .join(
        erp_metrics,
        "customer_unique_id",
        "left"
    )

)

# Avoid an extra Spark action before the single toPandas() conversion.

In [0]:
# ==========================================================
# ERP Dataset (optimized)
# ==========================================================

erp_pd = erp_base.toPandas()

ACCOUNT_MANAGERS = [

    "Carlos Souza",
    "Ana Lima",
    "Juliana Costa",
    "Pedro Oliveira",
    "Marcos Santos",
    "Fernanda Rocha"

]

CUSTOMER_GROUP = [

    "Retail",

    "SMB",

    "Corporate",

    "Government"

]

PAYMENT_TERMS = [

    "NET15",

    "NET30",

    "NET45",

    "NET60"

]

TAX_CATEGORY = [

    "Normal",

    "Simples",

    "MEI"

]

erp_rows = []

for c in erp_pd.itertuples(index=False):

    erp_rows.append({

        "erp_customer_id":

            c.erp_customer_id,

        "customer_unique_id":

            c.customer_unique_id,

        "customer_name":

            random_name(

                c.first_name,

                c.middle_name,

                c.last_name

            ),

        "cpf":

            c.cpf,

        "tax_category":

            random.choice(TAX_CATEGORY),

        "customer_group":

            random.choice(CUSTOMER_GROUP),

        "payment_terms":

            random.choice(PAYMENT_TERMS),

        "credit_limit":

            random.randint(

                3000,

                250000

            ),

        "account_manager":

            random.choice(ACCOUNT_MANAGERS),

        "invoice_email":

            random_email(

                c.first_name,

                c.last_name

            ),

        "total_orders":

            c.total_orders,

        "first_purchase":

            c.first_purchase,

        "last_purchase":

            c.last_purchase,

        "total_spent":

            c.total_spent,

        "average_ticket":

            c.average_ticket

    })

erp_pd = pd.DataFrame(erp_rows)  # generated once

In [0]:
# ==========================================================
# Duplicate ERP generation removed
# ==========================================================

# erp_pd was generated by the preceding cell; avoid a second Spark action.
# The duplicated implementation previously repeated toPandas() and all Faker work.
"""
erp_pd = erp_base.toPandas()

ACCOUNT_MANAGERS = [

    "Carlos Souza",
    "Ana Lima",
    "Juliana Costa",
    "Pedro Oliveira",
    "Marcos Santos",
    "Fernanda Rocha"

]

CUSTOMER_GROUP = [

    "Retail",

    "SMB",

    "Corporate",

    "Government"

]

PAYMENT_TERMS = [

    "NET15",

    "NET30",

    "NET45",

    "NET60"

]

TAX_CATEGORY = [

    "Normal",

    "Simples",

    "MEI"

]

erp_rows = []

for c in erp_pd.itertuples(index=False):

    erp_rows.append({

        "erp_customer_id":

            c.erp_customer_id,

        "customer_unique_id":

            c.customer_unique_id,

        "customer_name":

            random_name(

                c.first_name,

                c.middle_name,

                c.last_name

            ),

        "cpf":

            c.cpf,

        "tax_category":

            random.choice(TAX_CATEGORY),

        "customer_group":

            random.choice(CUSTOMER_GROUP),

        "payment_terms":

            random.choice(PAYMENT_TERMS),

        "credit_limit":

            random.randint(

                3000,

                250000

            ),

        "account_manager":

            random.choice(ACCOUNT_MANAGERS),

        "invoice_email":

            random_email(

                c.first_name,

                c.last_name

            ),

        "total_orders":

            c.total_orders,

        "first_purchase":

            c.first_purchase,

        "last_purchase":

            c.last_purchase,

        "total_spent":

            c.total_spent,

        "average_ticket":

            c.average_ticket

    })

erp_pd = pd.DataFrame(erp_rows)
"""

In [0]:
# ==========================================================
# ERP Data Quality
# ==========================================================

for i in erp_pd.index:

    erp_pd.at[i,"customer_name"] = inject_name_issue(

        erp_pd.at[i,"customer_name"]

    )

    erp_pd.at[i,"invoice_email"] = maybe_null(

        erp_pd.at[i,"invoice_email"],

        0.10

    )

    erp_pd.at[i,"credit_limit"] = maybe_null(

        erp_pd.at[i,"credit_limit"],

        0.08

    )

In [0]:
# ==========================================================
# ERP Spark
# ==========================================================

erp = spark.createDataFrame(erp_pd)

print()

print("ERP Rows")

print(erp.count())

display(erp.limit(10))

In [0]:
# ==========================================================
# ERP Business Rules
# ==========================================================

def calculate_credit_limit(total_spent, total_orders):

    if total_spent is None or pd.isna(total_spent):
        total_spent = 0

    if total_orders is None:
        total_orders = 0

    multiplier = random.uniform(2.0, 5.0)

    credit = max(
        3000,
        total_spent * multiplier
    )

    if total_orders > 20:
        credit *= 1.5

    return min(round(credit,2),250000)


def customer_group(total_spent):

    if total_spent is None:
        return "Prospect"

    if total_spent < 1000:
        return "Retail"

    if total_spent < 5000:
        return "Silver"

    if total_spent < 15000:
        return "Gold"

    return "Corporate"


def payment_terms(group):

    mapping = {

        "Prospect":"PREPAID",

        "Retail":"NET15",

        "Silver":"NET30",

        "Gold":"NET45",

        "Corporate":"NET60"

    }

    return mapping[group]


def choose_manager(group):

    managers = {

        "Prospect":[
            "Lucas Rocha",
            "Marina Costa"
        ],

        "Retail":[
            "Pedro Lima",
            "Juliana Alves"
        ],

        "Silver":[
            "Carlos Souza",
            "Fernanda Martins"
        ],

        "Gold":[
            "Ricardo Gomes",
            "Tatiane Melo"
        ],

        "Corporate":[
            "Eduardo Ribeiro",
            "Patricia Araujo"
        ]

    }

    return random.choice(managers[group])

In [0]:
# ==========================================================
# Ecommerce Reference Data
# ==========================================================

BROWSERS = [
    "Chrome",
    "Firefox",
    "Edge",
    "Safari",
    "Opera"
]

OPERATING_SYSTEMS = [
    "Windows",
    "Android",
    "iOS",
    "MacOS",
    "Linux"
]

DEVICES = [
    "Desktop",
    "Mobile",
    "Tablet"
]

MARKETING_SOURCE = [
    "Google",
    "Facebook",
    "Instagram",
    "Organic",
    "Referral",
    "LinkedIn",
    "TikTok"
]

LANGUAGES = [
    "pt-BR",
    "en-US",
    "es-ES"
]

In [0]:
# ==========================================================
# Favorite Product Category
# ==========================================================

favorite_category = (

    orders

    .join(order_items,"order_id")

    .join(products,"product_id")

    .join(
        customers.select(
            "customer_id",
            "customer_unique_id"
        ),
        "customer_id"
    )

    .groupBy(
        "customer_unique_id",
        "product_category_name"
    )

    .count()

)

window = Window.partitionBy(
    "customer_unique_id"
).orderBy(
    F.desc("count")
)

favorite_category = (

    favorite_category

    .withColumn(
        "rn",
        F.row_number().over(window)
    )

    .filter("rn=1")

    .drop("rn","count")

)

display(favorite_category.limit(10))

In [0]:
# ==========================================================
# Ecommerce Base
# ==========================================================

web_base = (

    ecommerce_seed

    .join(
        favorite_category,
        "customer_unique_id",
        "left"
    )

)

web_pd = web_base.toPandas()

In [0]:
# ==========================================================
# Ecommerce Dataset
# ==========================================================

print("=" * 80)
print("Generating Ecommerce...")
print("=" * 80)

web_rows = []

for c in web_pd.itertuples(index=False):

    display_name = random_name(
        c.first_name,
        c.middle_name,
        c.last_name
    )

    username = fake.user_name()

    browser = random.choice(BROWSERS)

    operating_system = random.choice(OPERATING_SYSTEMS)

    device = random.choice(DEVICES)

    marketing = random.choice(MARKETING_SOURCE)

    language = random.choice(LANGUAGES)

    newsletter = random.random() < 0.65

    account_created = fake.date_between(
        start_date="-8y",
        end_date="-30d"
    )

    last_login = fake.date_time_between(
        start_date="-30d",
        end_date="now"
    )

    wishlist = random.randint(0,40)

    carts_abandoned = random.randint(0,15)

    web_rows.append({

        "ecommerce_customer_id": c.ecommerce_customer_id,

        "customer_unique_id": c.customer_unique_id,

        "username": username,

        "display_name": display_name,

        "email_address": random_email(
            c.first_name,
            c.last_name
        ),

        "phone_number": phone_variation(c.phone),

        "browser": browser,

        "operating_system": operating_system,

        "device": device,

        "marketing_source": marketing,

        "preferred_language": language,

        "newsletter_optin": newsletter,

        "preferred_category": c.product_category_name,

        "wishlist_items": wishlist,

        "abandoned_carts": carts_abandoned,

        "account_created": account_created,

        "last_login": last_login

    })

web_pd = pd.DataFrame(web_rows)

print(web_pd.head())

In [0]:
# ==========================================================
# Ecommerce Data Quality
# ==========================================================

for i in web_pd.index:

    web_pd.at[i,"display_name"] = inject_name_issue(
        web_pd.at[i,"display_name"]
    )

    web_pd.at[i,"email_address"] = inject_email_issue(
        web_pd.at[i,"email_address"]
    )

    web_pd.at[i,"phone_number"] = maybe_null(
        web_pd.at[i,"phone_number"],
        0.08
    )

    if random.random() < 0.05:

        web_pd.at[i,"preferred_category"] = None

In [0]:
# ==========================================================
# Ecommerce Spark
# ==========================================================

web = spark.createDataFrame(web_pd)

print()

print("Ecommerce Customers")

print(web.count())

display(web.limit(10))

In [0]:
# ==========================================================
# Ecommerce Products
# ==========================================================

products_pd = products.toPandas()

BRANDS = [

    "Samsung",
    "LG",
    "Apple",
    "Philips",
    "Bosch",
    "Electrolux",
    "Dell",
    "Lenovo",
    "Nike",
    "Adidas",
    "Tramontina"

]

for i in products_pd.index:

    products_pd.at[i,"brand"] = random.choice(BRANDS)

    products_pd.at[i,"supplier_code"] = f"SUP-{random.randint(1000,9999)}"

    products_pd.at[i,"is_active"] = random.random() > 0.03

    products_pd.at[i,"launch_year"] = random.randint(2018,2024)

products_web = spark.createDataFrame(products_pd)

In [0]:
# ==========================================================
# Ecommerce Sellers
# ==========================================================

sellers_pd = sellers.toPandas()

for i in sellers_pd.index:

    sellers_pd.at[i,"seller_rating"] = round(
        random.uniform(3.5,5.0),
        2
    )

    sellers_pd.at[i,"premium_partner"] = (
        random.random() < 0.25
    )

    sellers_pd.at[i,"account_manager"] = random.choice([

        "Carlos Souza",

        "Ana Lima",

        "Fernanda Costa",

        "Pedro Santos"

    ])

products_web.printSchema()

sellers_web = spark.createDataFrame(sellers_pd)

In [0]:
# ==========================================================
# Loyalty Business Rules
# ==========================================================

def calculate_points(total_spent):

    if total_spent is None or pd.isna(total_spent):
        return 0

    return int(total_spent * 10)


def redeemed_points(points):

    if points == 0:
        return 0

    percentage = random.uniform(0.10, 0.70)

    return int(points * percentage)


def loyalty_tier(points):

    if points < 1000:
        return "Bronze"

    if points < 5000:
        return "Silver"

    if points < 15000:
        return "Gold"

    return "Platinum"


def wallet_balance(points):

    return round(points * 0.01,2)

In [0]:
# ==========================================================
# Loyalty Base
# ==========================================================

loyalty_base = (

    loyalty_seed

    .join(
        erp_metrics,
        "customer_unique_id",
        "left"
    )

)

# Avoid an extra Spark action before the single toPandas() conversion.

loyalty_pd = loyalty_base.toPandas()

In [0]:
# ==========================================================
# Loyalty Dataset
# ==========================================================

PREFERRED_STORE = [

    "São Paulo",

    "Rio de Janeiro",

    "Curitiba",

    "Belo Horizonte",

    "Brasília"

]

CAMPAIGNS = [

    "Black Friday",

    "Summer Sale",

    "Birthday",

    "Cashback",

    "Christmas"

]

loyalty_rows = []

for c in loyalty_pd.itertuples(index=False):

    points = calculate_points(
        c.total_spent
    )

    redeemed = redeemed_points(
        points
    )

    loyalty_rows.append({

        "loyalty_customer_id":

            c.loyalty_customer_id,

        "customer_unique_id":

            c.customer_unique_id,

        "member_name":

            random_name(

                c.first_name,

                c.middle_name,

                c.last_name

            ),

        "join_date":

            fake.date_between(

                start_date="-8y",

                end_date="-30d"

            ),

        "points":

            points,

        "redeemed_points":

            redeemed,

        "available_points":

            points - redeemed,

        "tier":

            loyalty_tier(points),

        "wallet_balance":

            wallet_balance(points),

        "preferred_store":

            random.choice(

                PREFERRED_STORE

            ),

        "last_campaign":

            random.choice(

                CAMPAIGNS

            ),

        "active_member":

            random.random() < 0.95

    })

loyalty_pd = pd.DataFrame(loyalty_rows)

In [0]:
# ==========================================================
# Loyalty Data Quality
# ==========================================================

for i in loyalty_pd.index:

    loyalty_pd.at[i,"member_name"] = inject_name_issue(

        loyalty_pd.at[i,"member_name"]

    )

    if random.random() < 0.03:

        loyalty_pd.at[i,"preferred_store"] = None

    if random.random() < 0.02:

        loyalty_pd.at[i,"wallet_balance"] = None

In [0]:
# ==========================================================
# Loyalty Spark
# ==========================================================

loyalty = spark.createDataFrame(loyalty_pd)

print()

print("Loyalty Members")

print(loyalty.count())

display(loyalty.limit(10))

In [0]:
# ==========================================================
# Validation
# ==========================================================

datasets = {

    "CRM": crm,

    "ERP": erp,

    "WEB": web,

    "LOYALTY": loyalty

}

summary = []

for name, df in datasets.items():

    summary.append(

        (

            name,

            df.count(),

            len(df.columns)

        )

    )

display(

    spark.createDataFrame(

        summary,

        [

            "system",

            "rows",

            "columns"

        ]

    )

)

In [0]:
# ==========================================================
# Write Landing
# ==========================================================

def repair_utf8_mojibake(df):
    """Repair text decoded as Latin-1 while preserving valid UTF-8 values."""
    markers = "|".join(chr(codepoint) for codepoint in (0x00C3, 0x00C2, 0x00E2))

    for field in df.schema.fields:
        if field.dataType.simpleString() != "string":
            continue

        value = F.col(field.name)
        repaired = F.decode(F.encode(value, "ISO-8859-1"), "UTF-8")
        df = df.withColumn(
            field.name,
            F.when(value.rlike(markers), repaired).otherwise(value)
        )

    return df

datasets = {

    OUTPUT_PATHS["crm_customers"]: crm.drop("customer_unique_id"),

    OUTPUT_PATHS["erp_customers"]: erp.drop("customer_unique_id"),

    OUTPUT_PATHS["erp_orders"]: orders,

    OUTPUT_PATHS["erp_payments"]: payments,

    OUTPUT_PATHS["ecommerce_customers"]: web.drop("customer_unique_id"),

    OUTPUT_PATHS["ecommerce_products"]: products_web,

    OUTPUT_PATHS["ecommerce_sellers"]: sellers_web,

    OUTPUT_PATHS["loyalty_customers"]: loyalty.drop("customer_unique_id")

}

for path, df in datasets.items():

    print(f"Writing {path}")

    clean_df = repair_utf8_mojibake(df)

    (

        clean_df.write

        .mode("overwrite")

        .parquet(path)

    )

print()

print("Landing successfully generated.")